# M1C2 - Topic 8: Data Visualization with Matplotlib and Seaborn

In Topic 7 you analyzed the FRIENDS dataset with pandas. Every result was
text-based — tables and print statements. In this topic we turn the **same
analyses** into **visual charts** using two industry-standard Python libraries:

- **Matplotlib** — the foundational plotting library. Almost every chart you
  see in Python is built on top of it.
- **Seaborn** — a higher-level library built on Matplotlib that produces
  beautiful statistical graphics with less code.

## Notebook Content

1. **Setup & Data Preparation** — Load the same FRIENDS dataset and reuse the Topic 7 pipeline
2. **Bar Charts** — Episodes per season, top directors, most popular episodes
3. **Line Charts** — Viewership and rating trends across seasons
4. **Histograms** — Summary length distribution
5. **Heatmaps** — Director × season table, confusion matrix
6. **Pie & Stacked Bar Charts** — Theme distributions
7. **Word Frequency Charts** — Top words overall and per genre
8. **Evaluation Charts** — Precision, recall, and classifier comparison

## Rationale

> *"A picture is worth a thousand words"*

Text tables are precise, but charts reveal **patterns, trends, and outliers**
at a glance. In Data Science, visualization is used at every stage:

| Stage | Purpose |
|---|---|
| **Exploration** | Spot distributions, outliers, missing data |
| **Analysis** | Compare groups, reveal correlations |
| **Communication** | Present findings to stakeholders |
| **Evaluation** | Visualize model performance (confusion matrix, ROC curves) |

Every chart in this notebook corresponds to a **text-based analysis from
Topic 7**, so you can see exactly what visualization adds.

---

## 1. Setup & Data Preparation

We reuse the **exact same data pipeline** from Topic 7: download the FRIENDS
dataset from Kaggle, clean it, preprocess the text, and classify every episode.

In [ ]:
!pip install --quiet kagglehub pandas matplotlib seaborn

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import os
import re
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.1)

CATEGORY_COLORS = {
    "Romance": "#e74c3c",
    "Career": "#3498db",
    "Family": "#2ecc71",
    "Friendship": "#f39c12",
}
CATEGORY_ORDER = ["Romance", "Career", "Family", "Friendship"]

### Load and clean the dataset (same as Topic 7)

In [ ]:
path = kagglehub.dataset_download("ruchi798/friends-tv-show-all-seasons-and-episodes-data")
csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]
df = pd.read_csv(os.path.join(path, csv_files[0]))

df = df[~df["Episode"].str.contains("Special", na=False)].copy()
df["Episode"] = df["Episode"].str.split("\n").str[0]
df["season"] = df["Episode"].str.split("-").str[0].astype(int)
df["episode_num"] = df["Episode"].str.split("-").str[1].astype(int)
df["viewers_millions"] = (
    df["U.S. viewers"].str.replace(" million", "", regex=False)
    .apply(pd.to_numeric, errors="coerce")
)
df["rating"] = (
    df["Rating/Share"].str.split("/").str[0]
    .apply(pd.to_numeric, errors="coerce")
)
df = df.dropna(subset=["Summary"])
df["summary_length"] = df["Summary"].str.len()

print(f"Loaded and cleaned: {len(df)} episodes, seasons {df['season'].min()}–{df['season'].max()}")

### Preprocess text and classify (same as Topic 7)

In [ ]:
STOP_WORDS = {
    "a", "an", "the", "and", "or", "but", "in", "on", "at", "to", "for",
    "of", "with", "by", "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did", "will", "would", "could",
    "should", "may", "might", "shall", "can", "not", "no", "nor",
    "it", "its", "he", "she", "they", "them", "his", "her", "their",
    "this", "that", "these", "those", "who", "whom", "which", "what",
    "when", "where", "how", "all", "each", "every", "both", "few",
    "more", "most", "other", "some", "such", "only", "own", "same",
    "than", "too", "very", "just", "about", "up", "out", "so", "if",
    "from", "into", "through", "during", "before", "after", "above",
    "as", "then", "once", "here", "there", "also", "over",
}

def preprocess(text):
    text = re.sub(r"[^\w\s]", "", text.lower())
    return [w for w in text.split() if w not in STOP_WORDS]

df["tokens"] = df["Summary"].apply(preprocess)

KEYWORDS = {
    "Romance": {
        "love", "kiss", "romance", "romantic", "couple", "feelings",
        "relationship", "dating", "girlfriend", "boyfriend", "engaged",
        "propose", "proposal", "devotion", "passionate", "confess",
        "reunite", "break", "heartbreak", "flirting",
    },
    "Career": {
        "job", "career", "work", "boss", "promotion", "office",
        "interview", "audition", "chef", "hire", "resume",
        "quit", "fired", "position", "business", "advertising",
        "stockbroker", "writer", "actor", "rejection",
    },
    "Family": {
        "baby", "pregnant", "family", "father", "mother", "dad",
        "mom", "parents", "birth", "child", "son", "daughter",
        "childhood", "wedding", "married", "ceremony", "traditions",
        "flashbacks", "house", "labor",
    },
    "Friendship": {
        "friends", "group", "gang", "together", "support", "bond",
        "loyalty", "trivia", "game", "apartment", "roommate",
        "competitive", "argue", "patience", "goodbye", "welcomes",
        "cooperates", "thanksgiving", "holiday", "park",
    },
}

def score_text(tokens, keyword_dict):
    token_set = set(tokens)
    return {cat: len(token_set & kw) for cat, kw in keyword_dict.items()}

def classify(tokens, keyword_dict):
    scores = score_text(tokens, keyword_dict)
    return max(scores, key=scores.get), scores

def score_text_tf(tokens, keyword_dict):
    return {cat: sum(1 for t in tokens if t in kw) for cat, kw in keyword_dict.items()}

def classify_tf(tokens, keyword_dict):
    scores = score_text_tf(tokens, keyword_dict)
    return max(scores, key=scores.get), scores

WEIGHTED_KEYWORDS = {
    "Romance": {
        "love": 3, "kiss": 3, "romance": 3, "romantic": 3, "passionate": 3,
        "couple": 2, "feelings": 2, "relationship": 2, "dating": 2,
        "confess": 2, "reunite": 2, "devotion": 2,
        "girlfriend": 1, "boyfriend": 1, "engaged": 1, "flirting": 1,
        "propose": 1, "proposal": 1, "break": 1, "heartbreak": 1,
    },
    "Career": {
        "job": 3, "career": 3, "work": 2, "boss": 2, "promotion": 2,
        "office": 2, "interview": 2, "audition": 2, "hire": 2,
        "chef": 2, "resume": 2, "quit": 2, "fired": 2,
        "position": 1, "business": 1, "advertising": 1,
        "stockbroker": 1, "writer": 1, "actor": 1, "rejection": 1,
    },
    "Family": {
        "baby": 3, "pregnant": 3, "family": 3, "birth": 3, "labor": 3,
        "father": 2, "mother": 2, "dad": 2, "mom": 2, "parents": 2,
        "child": 2, "son": 2, "daughter": 2, "childhood": 2,
        "wedding": 1, "married": 1, "ceremony": 1, "traditions": 1,
        "flashbacks": 1, "house": 1,
    },
    "Friendship": {
        "friends": 3, "group": 3, "gang": 3, "together": 2, "support": 2,
        "bond": 2, "loyalty": 2, "roommate": 2,
        "trivia": 1, "game": 1, "apartment": 1, "competitive": 1,
        "argue": 1, "patience": 1, "goodbye": 1, "welcomes": 1,
        "cooperates": 1, "thanksgiving": 1, "holiday": 1, "park": 1,
    },
}

def classify_weighted(tokens, wkw):
    scores = {}
    for cat, kw in wkw.items():
        scores[cat] = sum(kw.get(t, 0) for t in tokens)
    return max(scores, key=scores.get), scores

df["predicted"] = df["tokens"].apply(lambda t: classify(t, KEYWORDS)[0])
df["predicted_tf"] = df["tokens"].apply(lambda t: classify_tf(t, KEYWORDS)[0])
df["predicted_w"] = df["tokens"].apply(lambda t: classify_weighted(t, WEIGHTED_KEYWORDS)[0])

GROUND_TRUTH = {
    (1,1):"Friendship",(1,2):"Family",(1,7):"Romance",(1,15):"Career",
    (1,18):"Career",(1,24):"Romance",(2,5):"Friendship",(2,7):"Romance",
    (2,14):"Romance",(3,2):"Friendship",(3,9):"Friendship",(3,15):"Romance",
    (4,3):"Career",(4,12):"Friendship",(5,8):"Family",(5,14):"Romance",
    (6,6):"Friendship",(6,15):"Career",(6,24):"Romance",(7,17):"Career",
    (8,1):"Family",(8,23):"Family",(9,7):"Family",(10,12):"Family",
    (10,17):"Romance",
}
df["category"] = df.apply(lambda r: GROUND_TRUTH.get((r["season"], r["episode_num"])), axis=1)
df_labeled = df[df["category"].notna()].copy()

print(f"Pipeline complete: {len(df)} episodes classified, {len(df_labeled)} labeled for evaluation.")

---

## 2. Bar Charts — Counts and Rankings

Bar charts are the workhorse of data visualization. They map a **categorical
variable** (e.g. season number) to a **numeric value** (e.g. episode count).

### 2.1 Episodes per Season

*Topic 7 equivalent:* `df["season"].value_counts().sort_index()`

In [ ]:
eps_per_season = df["season"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(eps_per_season.index, eps_per_season.values, color="#5dade2", edgecolor="white")
ax.set_xlabel("Season")
ax.set_ylabel("Number of Episodes")
ax.set_title("Episodes per Season")
ax.set_xticks(range(1, 11))

for i, v in enumerate(eps_per_season.values):
    ax.text(eps_per_season.index[i], v + 0.3, str(v), ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

### 2.2 Top Directors

*Topic 7 equivalent:* `df["Directed by"].value_counts().head()`

A **horizontal bar chart** works well when category labels are long strings.

In [ ]:
top_dirs = df["Directed by"].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top_dirs.index[::-1], top_dirs.values[::-1], color="#48c9b0", edgecolor="white")
ax.set_xlabel("Number of Episodes Directed")
ax.set_title("Top 10 Directors by Episode Count")

for i, v in enumerate(top_dirs.values[::-1]):
    ax.text(v + 0.3, i, str(v), va="center", fontweight="bold")

plt.tight_layout()
plt.show()

### 2.3 Most Popular Episodes

*Topic 7 equivalent:* `popular.sort_values("viewers_millions", ascending=False)`

In [ ]:
top_eps = df.nlargest(15, "viewers_millions")
labels = [
    f"S{r['season']:02d}E{r['episode_num']:02d} {r['Title'][:30]}"
    for _, r in top_eps.iterrows()
]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(
    labels[::-1], top_eps["viewers_millions"].values[::-1],
    color="#f1948a", edgecolor="white",
)
ax.set_xlabel("U.S. Viewers (millions)")
ax.set_title("Top 15 Most-Watched Episodes")

for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{w:.1f}M", va="center", fontsize=9)

plt.tight_layout()
plt.show()

---

## 3. Line Charts — Trends Over Time

Line charts are ideal for showing how a metric **evolves** across an ordered
variable like season number or episode number.

### 3.1 Average Viewership by Season

*Topic 7 equivalent:* `df.groupby("season")["viewers_millions"].agg(["mean", "min", "max"])`

In [ ]:
stats = df.groupby("season")["viewers_millions"].agg(["mean", "min", "max"])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(stats.index, stats["mean"], "o-", color="#2e86c1", linewidth=2, label="Mean")
ax.fill_between(stats.index, stats["min"], stats["max"], alpha=0.2, color="#2e86c1", label="Min–Max range")
ax.set_xlabel("Season")
ax.set_ylabel("U.S. Viewers (millions)")
ax.set_title("Viewership Trends Across Seasons")
ax.set_xticks(range(1, 11))
ax.legend()

plt.tight_layout()
plt.show()

### 3.2 Average Rating by Season

*Topic 7 equivalent:* `df.groupby("season")["rating"].mean().sort_values(ascending=False)`

In [ ]:
rating_stats = df.groupby("season")["rating"].agg(["mean", "min", "max"])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(rating_stats.index, rating_stats["mean"], "s-", color="#e74c3c", linewidth=2, label="Mean")
ax.fill_between(rating_stats.index, rating_stats["min"], rating_stats["max"],
                alpha=0.15, color="#e74c3c", label="Min–Max range")
ax.set_xlabel("Season")
ax.set_ylabel("Rating")
ax.set_title("Rating Trends Across Seasons")
ax.set_xticks(range(1, 11))
ax.legend()

plt.tight_layout()
plt.show()

### 3.3 Viewership vs Rating — Dual Axis

We can overlay both metrics on the same chart using a **secondary y-axis**
to see if viewership and rating follow the same trend.

In [ ]:
viewer_avg = df.groupby("season")["viewers_millions"].mean()
rating_avg = df.groupby("season")["rating"].mean()

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.bar(viewer_avg.index - 0.15, viewer_avg.values, 0.3, color="#5dade2", alpha=0.8, label="Avg Viewers (M)")
ax2.plot(rating_avg.index, rating_avg.values, "D-", color="#e74c3c", linewidth=2, markersize=7, label="Avg Rating")

ax1.set_xlabel("Season")
ax1.set_ylabel("Avg Viewers (millions)", color="#5dade2")
ax2.set_ylabel("Avg Rating", color="#e74c3c")
ax1.set_title("Viewership vs Rating by Season")
ax1.set_xticks(range(1, 11))

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

plt.tight_layout()
plt.show()

---

## 4. Histograms — Distribution of Continuous Variables

A histogram shows how values are **distributed** across ranges (bins).
It answers: *"How many episodes have summaries of length 100–150 characters?"*

### 4.1 Summary Length Distribution

*Topic 7 equivalent:* `pd.crosstab(df["length_bin"], df["season"])`

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df["summary_length"], bins=25, color="#a569bd", edgecolor="white", alpha=0.85)
ax.axvline(df["summary_length"].mean(), color="#e74c3c", linestyle="--", linewidth=2,
           label=f"Mean = {df['summary_length'].mean():.0f} chars")
ax.axvline(df["summary_length"].median(), color="#2ecc71", linestyle="--", linewidth=2,
           label=f"Median = {df['summary_length'].median():.0f} chars")
ax.set_xlabel("Summary Length (characters)")
ax.set_ylabel("Number of Episodes")
ax.set_title("Distribution of Episode Summary Lengths")
ax.legend()

plt.tight_layout()
plt.show()

### 4.2 Summary Length by Season (Box Plot)

A **box plot** shows the median, quartiles, and outliers for each group —
much richer than just the mean.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x="season", y="summary_length", palette="Set2", ax=ax)
ax.set_xlabel("Season")
ax.set_ylabel("Summary Length (characters)")
ax.set_title("Summary Length Distribution by Season")

plt.tight_layout()
plt.show()

---

## 5. Heatmaps — Two-Dimensional Frequency Tables

A heatmap uses **color intensity** to represent values in a matrix.
It is the visual equivalent of `pd.crosstab()` or a pivot table.

### 5.1 Director × Season Heatmap

*Topic 7 equivalent:* `director_seasons.loc[top_directors]`

In [ ]:
director_seasons = df.groupby(["Directed by", "season"]).size().unstack(fill_value=0)
top_directors = df["Directed by"].value_counts().head(8).index
data = director_seasons.loc[top_directors]

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(data, annot=True, fmt="d", cmap="YlOrRd", linewidths=0.5, ax=ax)
ax.set_title("Episodes Directed per Season (Top 8 Directors)")
ax.set_xlabel("Season")
ax.set_ylabel("Director")

plt.tight_layout()
plt.show()

### 5.2 Viewership Heatmap by Season & Episode

This heatmap shows every episode's viewership as a color in a season × episode
grid — a quick way to spot the most-watched episodes.

In [ ]:
pivot = df.pivot_table(
    index="season", columns="episode_num", values="viewers_millions", aggfunc="first"
)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot, cmap="Blues", linewidths=0.3, ax=ax,
            cbar_kws={"label": "Viewers (millions)"})
ax.set_title("U.S. Viewership by Season and Episode")
ax.set_xlabel("Episode")
ax.set_ylabel("Season")

plt.tight_layout()
plt.show()

---

## 6. Word Frequency Charts

*Topic 7 equivalent:* `df["tokens"].explode().value_counts().head(20)` and
per-season word counts.

### 6.1 Top 20 Words Across All Episodes

In [ ]:
word_freq = df["tokens"].explode().value_counts()
top20 = word_freq.head(20)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top20.index[::-1], top20.values[::-1], color="#5dade2", edgecolor="white")
ax.set_xlabel("Frequency")
ax.set_title("Top 20 Most Frequent Words in Episode Summaries")

for i, v in enumerate(top20.values[::-1]):
    ax.text(v + 0.5, i, str(v), va="center", fontsize=9)

plt.tight_layout()
plt.show()

### 6.2 Top Words by Predicted Theme

Each subplot shows the most frequent words in episodes assigned to a
particular theme — revealing which vocabulary drives each classification.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, cat in zip(axes.flat, CATEGORY_ORDER):
    cat_tokens = df[df["predicted"] == cat]["tokens"].explode()
    top = cat_tokens.value_counts().head(10)
    ax.barh(top.index[::-1], top.values[::-1],
            color=CATEGORY_COLORS[cat], edgecolor="white")
    ax.set_title(cat, fontsize=13, fontweight="bold")
    ax.set_xlabel("Frequency")

plt.suptitle("Top 10 Words per Predicted Theme", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

---

## 7. Classification Results

*Topic 7 equivalent:* `df["predicted"].value_counts()` and
`pd.crosstab(df["season"], df["predicted"])`

### 7.1 Predicted Theme Distribution (Pie Chart)

A pie chart shows the **proportion** each category represents of the whole.

In [ ]:
theme_counts = df["predicted"].value_counts().reindex(CATEGORY_ORDER)
colors = [CATEGORY_COLORS[c] for c in CATEGORY_ORDER]

fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    theme_counts.values, labels=theme_counts.index, colors=colors,
    autopct="%1.1f%%", startangle=140, textprops={"fontsize": 12},
)
for t in autotexts:
    t.set_fontweight("bold")
ax.set_title("Predicted Theme Distribution (all 227 episodes)", fontsize=13)

plt.tight_layout()
plt.show()

### 7.2 Themes per Season (Stacked Bar Chart)

*Topic 7 equivalent:* `pd.crosstab(df["season"], df["predicted"])`

A stacked bar chart shows both the **total** per season and the **breakdown**
by theme.

In [ ]:
theme_season = pd.crosstab(df["season"], df["predicted"])[CATEGORY_ORDER]

fig, ax = plt.subplots(figsize=(10, 6))
theme_season.plot.bar(stacked=True, color=[CATEGORY_COLORS[c] for c in CATEGORY_ORDER],
                      edgecolor="white", ax=ax)
ax.set_xlabel("Season")
ax.set_ylabel("Number of Episodes")
ax.set_title("Predicted Themes per Season")
ax.legend(title="Theme", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.set_xticklabels(range(1, 11), rotation=0)

plt.tight_layout()
plt.show()

---

## 8. Evaluation Charts

Visualizing model performance makes it much easier to spot which categories
the classifier handles well and where it struggles.

### 8.1 Confusion Matrix Heatmap

*Topic 7 equivalent:* `pd.crosstab(df_labeled["category"], df_labeled["predicted"])`

The confusion matrix as a **heatmap** immediately shows where the classifier
confuses categories — dark cells off the diagonal indicate frequent errors.

In [ ]:
cm = pd.crosstab(
    df_labeled["category"], df_labeled["predicted"],
    rownames=["True"], colnames=["Predicted"],
)
cm = cm.reindex(index=CATEGORY_ORDER, columns=CATEGORY_ORDER, fill_value=0)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", linewidths=1,
            square=True, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Confusion Matrix — Set-Based Classifier", fontsize=13)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")

plt.tight_layout()
plt.show()

### 8.2 Precision and Recall (Grouped Bar Chart)

*Topic 7 equivalent:* `precision_recall_from_crosstab(cm)`

In [ ]:
def precision_recall_from_crosstab(cm):
    metrics = {}
    for label in cm.index:
        tp = cm.loc[label, label] if label in cm.columns else 0
        pred_total = cm[label].sum() if label in cm.columns else 0
        actual_total = cm.loc[label].sum()
        prec = tp / pred_total if pred_total > 0 else 0.0
        rec = tp / actual_total if actual_total > 0 else 0.0
        metrics[label] = {"Precision": prec, "Recall": rec}
    return pd.DataFrame(metrics).T

pr = precision_recall_from_crosstab(cm)
pr = pr.reindex(CATEGORY_ORDER)

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(CATEGORY_ORDER))
w = 0.3
ax.bar(x - w/2, pr["Precision"], w, label="Precision", color="#3498db", edgecolor="white")
ax.bar(x + w/2, pr["Recall"], w, label="Recall", color="#e67e22", edgecolor="white")

ax.set_xlabel("Category")
ax.set_ylabel("Score")
ax.set_title("Precision and Recall by Category")
ax.set_xticks(x)
ax.set_xticklabels(CATEGORY_ORDER)
ax.set_ylim(0, 1.15)
ax.legend()

for i in x:
    ax.text(i - w/2, pr["Precision"].iloc[i] + 0.03,
            f"{pr['Precision'].iloc[i]:.0%}", ha="center", fontsize=9, fontweight="bold")
    ax.text(i + w/2, pr["Recall"].iloc[i] + 0.03,
            f"{pr['Recall'].iloc[i]:.0%}", ha="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()

---

## 9. Classifier Comparison

*Topic 7 equivalent:* side-by-side accuracy table for Set-based, TF-based,
and Weighted classifiers.

### 9.1 Accuracy Comparison

In [ ]:
acc_set = (df_labeled["predicted"] == df_labeled["category"]).mean()
acc_tf = (
    df.loc[df["category"].notna(), "predicted_tf"]
    .eq(df.loc[df["category"].notna(), "category"]).mean()
)
acc_w = (
    df.loc[df["category"].notna(), "predicted_w"]
    .eq(df.loc[df["category"].notna(), "category"]).mean()
)

classifiers = ["Set-Based", "Term Frequency", "Weighted"]
accuracies = [acc_set, acc_tf, acc_w]
colors_clf = ["#3498db", "#e67e22", "#2ecc71"]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(classifiers, accuracies, color=colors_clf, edgecolor="white", width=0.5)
ax.set_ylabel("Accuracy")
ax.set_title("Classifier Accuracy Comparison (on 25 labeled episodes)")
ax.set_ylim(0, 1.15)

for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.03,
            f"{acc:.1%}", ha="center", fontweight="bold", fontsize=13)

plt.tight_layout()
plt.show()

### 9.2 Confusion Matrices — Side by Side

Comparing the three confusion matrices visually reveals which classifier
makes different types of errors.

In [ ]:
cm_set = pd.crosstab(df_labeled["category"], df_labeled["predicted"],
                     rownames=["True"], colnames=["Predicted"])
cm_tf = pd.crosstab(df_labeled["category"],
                    df.loc[df["category"].notna(), "predicted_tf"],
                    rownames=["True"], colnames=["Predicted"])
cm_w = pd.crosstab(df_labeled["category"],
                   df.loc[df["category"].notna(), "predicted_w"],
                   rownames=["True"], colnames=["Predicted"])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, matrix, title in zip(axes, [cm_set, cm_tf, cm_w],
                              ["Set-Based", "Term Frequency", "Weighted"]):
    matrix = matrix.reindex(index=CATEGORY_ORDER, columns=CATEGORY_ORDER, fill_value=0)
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", linewidths=1,
                square=True, cbar=False, ax=ax)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

plt.suptitle("Confusion Matrices — All Three Classifiers", fontsize=15, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

---

## Summary

In this notebook we took **every text-based analysis from Topic 7** and turned
it into a visual chart:

| Topic 7 Analysis | Topic 8 Visualization |
|---|---|
| `value_counts()` | Bar chart |
| `groupby().agg()` | Line chart with min/max range |
| `pd.crosstab()` (frequency table) | Heatmap |
| `value_counts()` on tokens | Horizontal bar chart |
| Predicted theme counts | Pie chart |
| Themes per season | Stacked bar chart |
| Confusion matrix | Heatmap with annotations |
| Precision / recall | Grouped bar chart |
| Accuracy comparison | Bar chart with labels |

### Key Takeaways

- **Matplotlib** gives you full control over every element of a chart
  (`fig`, `ax`, labels, ticks, colors, annotations).
- **Seaborn** builds on Matplotlib to produce polished statistical plots
  (`sns.heatmap`, `sns.boxplot`) with minimal code.
- The right chart type depends on the **data type**: bar for categories,
  line for trends, histogram for distributions, heatmap for matrices.
- Always add **titles**, **axis labels**, and **annotations** — a chart
  without labels is just a pretty shape.